# ثبت و تراز تصویر / Image Registration and Alignment

**هدف:** یادگیری روش‌های تراز کردن تصاویر و دوخت آن‌ها

**Objective:** Learn methods for aligning images and stitching them together

---

## محتوا / Contents:
1. مقدمه‌ای بر ثبت تصویر / Introduction to Image Registration
2. تطبیق الگو / Template Matching
3. همبستگی فاز / Phase Correlation
4. تراز مبتنی بر ویژگی / Feature-Based Alignment
5. دوخت تصاویر / Image Stitching
6. کاربردهای عملی / Practical Applications

In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List

# تنظیمات نمایش / Display settings
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. توابع کمکی / Helper Functions

In [ ]:
def show_comparison(images: list, titles: list, rows: int = 1, cols: int = 2, cmap: str = 'gray'):
    """
    نمایش مقایسه چند تصویر
    """
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    axes = axes.flatten() if rows * cols > 1 else [axes]
    
    for idx, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:
            axes[idx].imshow(img, cmap=cmap)
        else:
            axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(title, fontsize=14)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

## 2. تطبیق الگو (Template Matching)

**توضیح:** تطبیق الگو برای یافتن موقعیت یک تصویر کوچک (الگو) در تصویر بزرگ‌تر استفاده می‌شود.

**Explanation:** Template matching is used to find the location of a small image (template) within a larger image.

In [ ]:
# ایجاد تصویر تست
def create_scene_image() -> np.ndarray:
    """
    ایجاد تصویر صحنه با اشکال مختلف
    """
    img = np.ones((500, 700, 3), dtype=np.uint8) * 200
    
    # رسم اشکال مختلف
    cv2.rectangle(img, (50, 50), (150, 150), (255, 0, 0), -1)
    cv2.circle(img, (400, 100), 50, (0, 255, 0), -1)
    cv2.rectangle(img, (200, 300), (350, 450), (0, 0, 255), -1)
    
    # رسم ستاره (الگوی هدف)
    star_center = (550, 350)
    star_pts = []
    for i in range(5):
        angle = i * 144 * np.pi / 180
        x = int(star_center[0] + 40 * np.cos(angle))
        y = int(star_center[1] + 40 * np.sin(angle))
        star_pts.append([x, y])
    cv2.fillPoly(img, [np.array(star_pts, np.int32)], (255, 255, 0))
    
    return img


scene = create_scene_image()

# استخراج الگو (ستاره)
template = scene[310:390, 510:590].copy()

show_comparison([scene, template],
                ['صحنه / Scene', 'الگو / Template'],
                rows=1, cols=2)

print(f"اندازه صحنه / Scene size: {scene.shape[:2]}")
print(f"اندازه الگو / Template size: {template.shape[:2]}")

In [ ]:
# تبدیل به خاکستری
scene_gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)
template_gray = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

# اعمال تطبیق الگو با روش‌های مختلف
methods = [
    ('TM_CCOEFF', cv2.TM_CCOEFF),
    ('TM_CCOEFF_NORMED', cv2.TM_CCOEFF_NORMED),
    ('TM_CCORR', cv2.TM_CCORR),
    ('TM_CCORR_NORMED', cv2.TM_CCORR_NORMED),
    ('TM_SQDIFF', cv2.TM_SQDIFF),
    ('TM_SQDIFF_NORMED', cv2.TM_SQDIFF_NORMED)
]

results = []
for method_name, method in methods:
    # اعمال تطبیق الگو
    result = cv2.matchTemplate(scene_gray, template_gray, method)
    
    # یافتن بهترین تطابق
    min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)
    
    # برای روش‌های SQDIFF، کمترین مقدار بهترین است
    if method in [cv2.TM_SQDIFF, cv2.TM_SQDIFF_NORMED]:
        top_left = min_loc
        match_val = min_val
    else:
        top_left = max_loc
        match_val = max_val
    
    # رسم مستطیل دور الگوی یافت شده
    h, w = template_gray.shape
    bottom_right = (top_left[0] + w, top_left[1] + h)
    
    scene_copy = scene.copy()
    cv2.rectangle(scene_copy, top_left, bottom_right, (0, 255, 0), 3)
    
    results.append((method_name, scene_copy, match_val))
    print(f"{method_name}: موقعیت={top_left}, مقدار تطابق={match_val:.4f}")

# نمایش نتایج
images = [r[1] for r in results[:4]]
titles = [f"{r[0]}\nMatch: {r[2]:.3f}" for r in results[:4]]
show_comparison(images, titles, rows=2, cols=2)

## 3. تطبیق الگو با چرخش / Template Matching with Rotation

**توضیح:** تطبیق الگوی معمولی نسبت به چرخش حساس است. برای حل این مشکل باید الگو را در زوایای مختلف امتحان کنیم.

**Explanation:** Standard template matching is sensitive to rotation. To solve this, we need to try the template at different angles.

In [ ]:
def match_template_rotation(scene: np.ndarray, template: np.ndarray, 
                           angles: List[float] = None) -> Tuple[float, float, float]:
    """
    تطبیق الگو با امتحان زوایای مختلف
    
    Args:
        scene: تصویر صحنه
        template: الگو
        angles: لیست زوایا برای امتحان
    
    Returns:
        بهترین زاویه، موقعیت و مقدار تطابق
    """
    if angles is None:
        angles = range(0, 360, 15)
    
    scene_gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)
    template_gray = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)
    
    best_match = -1
    best_angle = 0
    best_loc = (0, 0)
    
    h, w = template_gray.shape
    center = (w // 2, h // 2)
    
    for angle in angles:
        # چرخش الگو
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(template_gray, M, (w, h))
        
        # تطبیق الگو
        result = cv2.matchTemplate(scene_gray, rotated, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, max_loc = cv2.minMaxLoc(result)
        
        if max_val > best_match:
            best_match = max_val
            best_angle = angle
            best_loc = max_loc
    
    return best_angle, best_loc, best_match


# ایجاد صحنه با الگوی چرخیده
scene_rotated = create_scene_image()
h, w = template.shape[:2]
center = (w // 2, h // 2)
M_rot = cv2.getRotationMatrix2D(center, 45, 1.0)
template_rotated = cv2.warpAffine(template, M_rot, (w, h))

# جایگذاری الگوی چرخیده در صحنه
scene_rotated[310:310+h, 510:510+w] = template_rotated

# تطبیق با چرخش
best_angle, best_loc, best_match = match_template_rotation(scene_rotated, template)

print(f"بهترین زاویه / Best angle: {best_angle}°")
print(f"موقعیت / Location: {best_loc}")
print(f"مقدار تطابق / Match value: {best_match:.4f}")

# رسم نتیجه
scene_result = scene_rotated.copy()
cv2.rectangle(scene_result, best_loc, (best_loc[0] + w, best_loc[1] + h), (0, 255, 0), 3)

show_comparison([template, template_rotated, scene_result],
                ['الگوی اصلی / Original Template',
                 f'الگوی چرخیده 45° / Rotated 45°',
                 f'یافت شده در زاویه {best_angle}° / Found at {best_angle}°'],
                rows=1, cols=3)

## 4. همبستگی فاز (Phase Correlation)

**توضیح:** همبستگی فاز برای یافتن جابجایی بین دو تصویر استفاده می‌شود. سریع‌تر از تطبیق الگو است.

**Explanation:** Phase correlation is used to find the translation between two images. It's faster than template matching.

In [ ]:
# ایجاد دو تصویر با جابجایی مشخص
img1 = create_scene_image()

# جابجایی تصویر
tx, ty = 50, 30
M_trans = np.float32([[1, 0, tx], [0, 1, ty]])
img2 = cv2.warpAffine(img1, M_trans, (img1.shape[1], img1.shape[0]))

# تبدیل به خاکستری
img1_gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY).astype(np.float32)
img2_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY).astype(np.float32)

# محاسبه همبستگی فاز
(shift_x, shift_y), response = cv2.phaseCorrelate(img1_gray, img2_gray)

print(f"جابجایی واقعی / Actual shift: ({tx}, {ty})")
print(f"جابجایی تشخیص داده شده / Detected shift: ({shift_x:.2f}, {shift_y:.2f})")
print(f"پاسخ / Response: {response:.4f}")

# اصلاح تصویر دوم
M_correction = np.float32([[1, 0, -shift_x], [0, 1, -shift_y]])
img2_aligned = cv2.warpAffine(img2, M_correction, (img2.shape[1], img2.shape[0]))

# محاسبه تفاوت
diff_before = cv2.absdiff(img1, img2)
diff_after = cv2.absdiff(img1, img2_aligned)

show_comparison([img1, img2, img2_aligned, diff_before, diff_after],
                ['تصویر 1 / Image 1',
                 'تصویر 2 (جابجا شده) / Image 2 (Shifted)',
                 'تصویر 2 (تراز شده) / Image 2 (Aligned)',
                 'تفاوت قبل / Diff Before',
                 'تفاوت بعد / Diff After'],
                rows=2, cols=3)

## 5. تراز مبتنی بر ویژگی (Feature-Based Alignment)

**توضیح:** استفاده از نقاط کلیدی و توصیفگرها برای تراز کردن تصاویر.

**Explanation:** Using keypoints and descriptors to align images.

In [ ]:
def align_images_features(img1: np.ndarray, img2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    تراز کردن تصاویر با استفاده از ویژگی‌ها
    
    Args:
        img1: تصویر مرجع
        img2: تصویر برای تراز کردن
    
    Returns:
        تصویر تراز شده و ماتریس تبدیل
    """
    # تبدیل به خاکستری
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    
    # تشخیص ویژگی‌ها با ORB
    orb = cv2.ORB_create(5000)
    kp1, des1 = orb.detectAndCompute(gray1, None)
    kp2, des2 = orb.detectAndCompute(gray2, None)
    
    # تطبیق ویژگی‌ها
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    matches = matcher.knnMatch(des1, des2, k=2)
    
    # فیلتر کردن تطابق‌های خوب (Lowe's ratio test)
    good_matches = []
    for m_n in matches:
        if len(m_n) == 2:
            m, n = m_n
            if m.distance < 0.75 * n.distance:
                good_matches.append(m)
    
    print(f"تعداد ویژگی‌های تصویر 1 / Features in image 1: {len(kp1)}")
    print(f"تعداد ویژگی‌های تصویر 2 / Features in image 2: {len(kp2)}")
    print(f"تعداد تطابق‌های خوب / Good matches: {len(good_matches)}")
    
    if len(good_matches) < 4:
        print("تطابق‌های کافی یافت نشد / Not enough matches found")
        return img2, None
    
    # استخراج نقاط متناظر
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    
    # محاسبه ماتریس هوموگرافی با RANSAC
    M, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
    
    if M is None:
        print("محاسبه هوموگرافی ناموفق / Homography computation failed")
        return img2, None
    
    # اعمال تبدیل
    h, w = img1.shape[:2]
    aligned = cv2.warpPerspective(img2, M, (w, h))
    
    # رسم تطابق‌ها
    matches_img = cv2.drawMatches(img1, kp1, img2, kp2, good_matches[:50], None,
                                  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    
    return aligned, matches_img


# ایجاد دو تصویر با تبدیلات مختلف
base_img = create_scene_image()

# اعمال چرخش و جابجایی
h, w = base_img.shape[:2]
center = (w // 2, h // 2)
M_transform = cv2.getRotationMatrix2D(center, 15, 0.9)
M_transform[0, 2] += 30
M_transform[1, 2] += 20
transformed_img = cv2.warpAffine(base_img, M_transform, (w, h))

# تراز کردن
aligned_img, matches_img = align_images_features(base_img, transformed_img)

if matches_img is not None:
    plt.figure(figsize=(15, 5))
    plt.imshow(cv2.cvtColor(matches_img, cv2.COLOR_BGR2RGB))
    plt.title('تطابق‌های ویژگی / Feature Matches')
    plt.axis('off')
    plt.show()

if aligned_img is not None:
    show_comparison([base_img, transformed_img, aligned_img],
                    ['تصویر مرجع / Reference',
                     'تصویر تبدیل شده / Transformed',
                     'تراز شده / Aligned'],
                    rows=1, cols=3)

## 6. دوخت تصاویر ساده (Simple Image Stitching)

**توضیح:** ترکیب چند تصویر برای ایجاد پانوراما.

**Explanation:** Combining multiple images to create a panorama.

In [ ]:
def create_panorama_images() -> List[np.ndarray]:
    """
    ایجاد سه تصویر برای دوخت پانوراما
    """
    # ایجاد یک صحنه بزرگ
    full_scene = np.ones((400, 1200, 3), dtype=np.uint8) * 150
    
    # اضافه کردن اشکال در موقعیت‌های مختلف
    cv2.rectangle(full_scene, (50, 100), (200, 300), (255, 0, 0), -1)
    cv2.circle(full_scene, (400, 200), 80, (0, 255, 0), -1)
    cv2.rectangle(full_scene, (600, 150), (750, 350), (0, 0, 255), -1)
    cv2.circle(full_scene, (950, 200), 70, (255, 255, 0), -1)
    cv2.rectangle(full_scene, (1000, 100), (1150, 300), (255, 0, 255), -1)
    
    # برش سه تصویر با همپوشانی
    img1 = full_scene[:, 0:500].copy()
    img2 = full_scene[:, 350:850].copy()
    img3 = full_scene[:, 700:1200].copy()
    
    return [img1, img2, img3]


# ایجاد تصاویر
images = create_panorama_images()

show_comparison(images,
                ['تصویر چپ / Left Image',
                 'تصویر وسط / Middle Image',
                 'تصویر راست / Right Image'],
                rows=1, cols=3)

# استفاده از Stitcher OpenCV
stitcher = cv2.Stitcher_create()
status, panorama = stitcher.stitch(images)

if status == cv2.Stitcher_OK:
    print("دوخت موفقیت‌آمیز بود / Stitching successful")
    plt.figure(figsize=(15, 5))
    plt.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
    plt.title('پانورامای دوخته شده / Stitched Panorama')
    plt.axis('off')
    plt.show()
else:
    print(f"دوخت ناموفق بود. کد خطا: {status} / Stitching failed. Error code: {status}")

## 7. کاربرد عملی: تراز تصاویر پزشکی / Practical: Medical Image Alignment

In [ ]:
def create_medical_image() -> np.ndarray:
    """
    ایجاد تصویر شبیه‌سازی شده پزشکی
    """
    img = np.zeros((400, 400), dtype=np.uint8)
    
    # شبیه‌سازی بافت
    img = cv2.randn(img, 100, 30)
    img = cv2.GaussianBlur(img, (15, 15), 0)
    
    # اضافه کردن ساختارها
    cv2.circle(img, (200, 200), 80, 180, -1)
    cv2.circle(img, (150, 150), 30, 220, -1)
    cv2.circle(img, (250, 250), 25, 200, -1)
    cv2.ellipse(img, (300, 150), (40, 60), 30, 0, 360, 160, -1)
    
    return img


# ایجاد دو تصویر پزشکی (مثلاً از دو زمان مختلف)
medical_img1 = create_medical_image()

# تصویر دوم با تغییرات جزئی
h, w = medical_img1.shape
center = (w // 2, h // 2)
M_medical = cv2.getRotationMatrix2D(center, 5, 1.02)
M_medical[0, 2] += 10
M_medical[1, 2] += 5
medical_img2 = cv2.warpAffine(medical_img1, M_medical, (w, h))

# اضافه کردن نویز متفاوت
noise = np.random.normal(0, 10, medical_img2.shape).astype(np.int16)
medical_img2 = np.clip(medical_img2.astype(np.int16) + noise, 0, 255).astype(np.uint8)

# تراز کردن با همبستگی فاز
(shift_x, shift_y), response = cv2.phaseCorrelate(
    medical_img1.astype(np.float32),
    medical_img2.astype(np.float32)
)

print(f"جابجایی تشخیص داده شده / Detected shift: ({shift_x:.2f}, {shift_y:.2f})")
print(f"پاسخ / Response: {response:.4f}")

# اصلاح تراز
M_correction = np.float32([[1, 0, -shift_x], [0, 1, -shift_y]])
medical_img2_aligned = cv2.warpAffine(medical_img2, M_correction, (w, h))

# محاسبه تفاوت
diff_before = cv2.absdiff(medical_img1, medical_img2)
diff_after = cv2.absdiff(medical_img1, medical_img2_aligned)

# ایجاد نقشه حرارتی از تفاوت
diff_heatmap = cv2.applyColorMap(diff_after, cv2.COLORMAP_JET)

show_comparison([medical_img1, medical_img2, medical_img2_aligned, 
                diff_before, diff_after, diff_heatmap],
                ['تصویر 1 / Image 1',
                 'تصویر 2 / Image 2',
                 'تصویر 2 تراز شده / Image 2 Aligned',
                 'تفاوت قبل / Diff Before',
                 'تفاوت بعد / Diff After',
                 'نقشه حرارتی تفاوت / Diff Heatmap'],
                rows=2, cols=3, cmap='gray')

## 8. تمرین‌ها / Exercises

### تمرین 1 / Exercise 1:
تابعی بنویسید که تطبیق الگو را با مقیاس‌های مختلف نیز امتحان کند (علاوه بر چرخش).

Write a function that tries template matching with different scales (in addition to rotation).

### تمرین 2 / Exercise 2:
از SIFT یا SURF به جای ORB برای تراز مبتنی بر ویژگی استفاده کنید و نتایج را مقایسه کنید.

Use SIFT or SURF instead of ORB for feature-based alignment and compare the results.

### تمرین 3 / Exercise 3:
یک برنامه دوخت پانوراما بسازید که بتواند 4 یا بیشتر تصویر را به هم بدوزد.

Build a panorama stitching program that can stitch 4 or more images together.

### تمرین 4 / Exercise 4:
تراز تصاویر را با استفاده از ECC (Enhanced Correlation Coefficient) پیاده‌سازی کنید.

Implement image alignment using ECC (Enhanced Correlation Coefficient).

**راهنمایی / Hint:** از `cv2.findTransformECC()` استفاده کنید.

In [ ]:
# فضای کار برای تمرین‌ها / Workspace for exercises

# تمرین 1 / Exercise 1
# کد خود را اینجا بنویسید


# تمرین 2 / Exercise 2
# کد خود را اینجا بنویسید


# تمرین 3 / Exercise 3
# کد خود را اینجا بنویسید


# تمرین 4 / Exercise 4
# کد خود را اینجا بنویسید